<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/02_deep_training/notebooks/sesion_02_busqueda_hiperparametros.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/02_deep_training/notebooks/sesion_02_busqueda_hiperparametros.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F02_deep_training%2Fnotebooks%2Fsesion_02_busqueda_hiperparametros.ipynb)

# SI7011 · Sesión 02 · Búsqueda de hiperparámetros con Optuna y MLflow
**Juan David Martínez Vargas · Universidad EAFIT**

En la Práctica B elegimos la receta **a mano**, una decisión a la vez. Aquí dejamos que un algoritmo busque la combinación de hiperparámetros y usamos **MLflow** para ver y comparar cada intento.

**Parámetros e hiperparámetros.** Los parámetros $\theta$ (pesos y sesgos) los aprende el gradiente. Los hiperparámetros (η, weight decay, Dropout, profundidad, ancho, tamaño de lote, normalización…) se fijan **antes** de entrenar y se eligen con el conjunto de **validación**:

$$
\lambda^\star=\arg\min_{\lambda}\;J_\text{val}\big(\theta^\star(\lambda)\big),
\qquad
\theta^\star(\lambda)=\arg\min_\theta J_\text{train}(\theta;\lambda)
$$

Cada evaluación de $\lambda$ cuesta un entrenamiento completo, así que el presupuesto se mide en **corridas**.

| Herramienta | Papel |
|---|---|
| **Optuna** | propone hiperparámetros (muestreador TPE) y detiene temprano los intentos poco prometedores (*pruning*) |
| **MLflow** | una corrida **padre** por estudio y una corrida **hija** por intento, con parámetros, curvas y estado |

**Datos:** Covertype, como en la Práctica B (mismo contrato y la misma partición).
**Duración:** 60 minutos. **Entorno:** Colab o Kaggle; GPU recomendada, CPU suficiente con los tamaños por defecto (≈ 15 min de búsqueda).

In [ ]:
%pip install -q mlflow optuna

In [ ]:
import copy, hashlib, json, math, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

import mlflow
import optuna

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '| MLflow', mlflow.__version__, '| Optuna', optuna.__version__, '| device:', device)

## 1. Datos: el mismo pipeline de la Práctica B · 5 min

Carga, contrato de datos, partición 60/20/20 estratificada y escalado de las 10 numéricas con estadísticas de entrenamiento. Para que la búsqueda quepa en clase usamos **30 000** ejemplos de entrenamiento; validación y prueba quedan completas.

In [ ]:
def load_covertype():
    roots = [Path('/kaggle/input'), Path('.')]
    hits = [p for r in roots if r.exists() for p in r.glob('**/covtype.csv')]
    if hits:
        print('Leyendo', hits[0])
        return pd.read_csv(hits[0]), hits[0]
    from sklearn.datasets import fetch_covtype
    print('covtype.csv no encontrado: descargando con sklearn.datasets.fetch_covtype')
    return fetch_covtype(as_frame=True).frame, None


df, src = load_covertype()
FEATURES = list(df.columns[:54]); NUMERIC = FEATURES[:10]; N_NUM = 10; C = 7
CONTRACT = {'features': FEATURES, 'numeric': NUMERIC, 'binary': FEATURES[10:], 'target': 'Cover_Type',
            'classes': list(range(1, C + 1))}


def validate_rows(frame, contract=CONTRACT):
    missing = [c for c in contract['features'] if c not in frame.columns]
    if missing:
        raise ValueError(f'Faltan columnas: {missing[:5]}')
    X = frame[contract['features']]
    if X.isna().any().any() or not all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns):
        raise ValueError('Valores faltantes o columnas no numéricas.')
    if not X[contract['binary']].isin([0, 1]).all().all():
        raise ValueError('Indicadores con valores distintos de 0/1.')
    return X.to_numpy(dtype=np.float32, copy=True)


X_all = validate_rows(df)
y_all = df.iloc[:, -1].to_numpy().astype(np.int64) - 1
data_card = {'fuente': str(src) if src else 'sklearn.datasets.fetch_covtype',
             'sha256': hashlib.sha256(src.read_bytes()).hexdigest()[:16] if src else None,
             'registros': len(df), 'clases': np.bincount(y_all, minlength=C).tolist()}
with open('data_card.json', 'w') as f:
    json.dump(data_card, f, indent=1)

TRAIN_SIZE = 30_000
X_rest, X_test, y_rest, y_test = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=SEED)
X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=TRAIN_SIZE, stratify=y_train, random_state=SEED)
scaler = StandardScaler().fit(X_train[:, :N_NUM])
for A in (X_train, X_val, X_test):
    A[:, :N_NUM] = scaler.transform(A[:, :N_NUM])
T = {k: torch.as_tensor(v, device=device) for k, v in
     dict(X_train=X_train, X_val=X_val, X_test=X_test, y_train=y_train, y_val=y_val, y_test=y_test).items()}
print('entrenamiento', X_train.shape, '| validación', X_val.shape, '| prueba', X_test.shape)

## 2. Entrenamiento configurable (listo) · 5 min

Es la receta de la Práctica B resumida en una función. Lo nuevo es el argumento `report`: después de cada época, `fit` llama `report(epoch, val_loss)`. Así un agente externo (Optuna) puede **observar** el entrenamiento y decidir detenerlo.

In [ ]:
BASE = dict(depth=6, width=256, norm='batchnorm', dropout=0.1, lr=1e-3, weight_decay=1e-2,
            batch_size=512, warmup_epochs=1, epochs=12, patience=3, clip_norm=1.0, seed=SEED)


def make_mlp(cfg, n_in=54, n_out=C):
    bn = cfg['norm'] == 'batchnorm'
    layers, d = [], n_in
    for _ in range(cfg['depth']):
        layers.append(nn.Linear(d, cfg['width'], bias=not bn))
        if bn:
            layers.append(nn.BatchNorm1d(cfg['width']))
        layers.append(nn.ReLU())
        if cfg['dropout'] > 0:
            layers.append(nn.Dropout(cfg['dropout']))
        d = cfg['width']
    layers.append(nn.Linear(d, n_out))
    model = nn.Sequential(*layers)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model


@torch.no_grad()
def evaluate(model, X, y, bs=8192):
    model.eval()
    logits = torch.cat([model(X[i:i + bs]) for i in range(0, len(y), bs)])
    pred = logits.argmax(1).cpu().numpy(); true = y.cpu().numpy()
    return {'loss': F.cross_entropy(logits, y).item(), 'acc': accuracy_score(true, pred),
            'macro_f1': f1_score(true, pred, average='macro', zero_division=0)}


def fit(cfg, report=None):
    set_seed(cfg['seed'])
    model = make_mlp(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    n = len(T['y_train']); spe = math.ceil(n / cfg['batch_size'])
    total, warm = cfg['epochs'] * spe, cfg['warmup_epochs'] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else
                                              0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    gen = torch.Generator().manual_seed(cfg['seed'])
    history, best, best_state, bad = [], {'val_loss': float('inf')}, None, 0
    for epoch in range(cfg['epochs']):
        model.train(); s = 0.0
        for idx in torch.randperm(n, generator=gen).split(cfg['batch_size']):
            idx = idx.to(device)
            loss = F.cross_entropy(model(T['X_train'][idx]), T['y_train'][idx])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip_norm'])
            opt.step(); sched.step(); s += loss.item() * len(idx)
        va = evaluate(model, T['X_val'], T['y_val'])
        row = {'epoch': epoch, 'train_loss': s / n, 'val_loss': va['loss'], 'val_acc': va['acc'],
               'val_macro_f1': va['macro_f1']}
        history.append(row)
        if not math.isfinite(row['val_loss']):
            break
        if row['val_loss'] < best['val_loss']:
            best, best_state, bad = row, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        if report is not None:
            report(epoch, row['val_loss'])          # puede lanzar optuna.TrialPruned
        if bad >= cfg['patience']:
            break
    return history, best, best_state


def log_history(history):
    for row in history:
        mlflow.log_metrics({k: v for k, v in row.items() if k != 'epoch'}, step=row['epoch'])

## 3. Línea base registrada en MLflow · 3 min

Antes de buscar, registramos la receta de referencia (la R5 de la Práctica B, ajustada a este tamaño). Toda búsqueda debe compararse contra una línea base **con el mismo presupuesto y los mismos datos**.

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
EXPERIMENT = 'S02-hparam-tuning'
mlflow.set_experiment(EXPERIMENT)

t0 = time.time()
with mlflow.start_run(run_name='linea-base'):
    mlflow.log_params(BASE)
    history, best_base, _ = fit(BASE)
    log_history(history)
    mlflow.log_metrics({'best_val_loss': best_base['val_loss'], 'best_val_macro_f1': best_base['val_macro_f1'],
                        'train_seconds': time.time() - t0})
print(f"línea base: val_loss {best_base['val_loss']:.4f} | F1 macro {best_base['val_macro_f1']:.3f} | "
      f"{time.time() - t0:.0f} s")

## 4. Espacio de búsqueda · TODO 1 · 10 min

Optuna pregunta por cada hiperparámetro con `trial.suggest_*`. Implementa `search_space(trial)` y devuelve un diccionario con:

| hiperparámetro | rango | escala |
|---|---|---|
| `lr` | $10^{-4}$ – $10^{-2}$ | **logarítmica** (`log=True`) |
| `weight_decay` | $10^{-6}$ – $10^{-1}$ | **logarítmica** |
| `dropout` | 0 – 0.5 | lineal |
| `depth` | {2, 4, 6, 8} | categórica |
| `width` | {128, 256, 512} | categórica |
| `batch_size` | {256, 512, 1024} | categórica |
| `norm` | {`'none'`, `'batchnorm'`} | categórica |

Usa `trial.suggest_float(nombre, bajo, alto, log=...)` y `trial.suggest_categorical(nombre, opciones)`.

In [ ]:
def search_space(trial):
    # TODO 1: devuelve {'lr': ..., 'weight_decay': ..., 'dropout': ..., 'depth': ..., 'width': ...,
    #                   'batch_size': ..., 'norm': ...}
    raise NotImplementedError('Completa TODO 1')

_t = optuna.trial.FixedTrial({'lr': 1e-3, 'weight_decay': 1e-4, 'dropout': 0.2, 'depth': 4, 'width': 256,
                             'batch_size': 512, 'norm': 'batchnorm'})
_cfg = search_space(_t)
assert set(_cfg) == {'lr', 'weight_decay', 'dropout', 'depth', 'width', 'batch_size', 'norm'}
print('Espacio de búsqueda OK:', _cfg)

**Pregunta 1.** ¿Por qué muestreamos `lr` y `weight_decay` en escala logarítmica? Con muestreo uniforme entre $10^{-4}$ y $10^{-2}$, ¿qué fracción de los intentos caería por debajo de $10^{-3}$?

_Respuesta:_

## 5. Objetivo con *pruning* y corridas anidadas en MLflow · TODO 2 · 10 min

Cada intento de Optuna es una corrida **hija** (`nested=True`) dentro de la corrida del estudio. La función `objective`:

1. pide hiperparámetros con `search_space` y los combina con `BASE`;
2. entrena con `fit(cfg, report=...)`;
3. registra parámetros, curvas y el estado final (`COMPLETE` o `PRUNED`);
4. devuelve la mejor `val_loss`, que Optuna minimiza.

**Pruning.** El *median pruner* detiene un intento si, en la misma época, su pérdida de validación es peor que la mediana de los intentos anteriores. Ahorra tiempo en configuraciones que ya se ven malas.

**TODO 2.** Completa `report(epoch, val_loss)` dentro de `objective`: informar el valor con `trial.report(val_loss, epoch)` y, si `trial.should_prune()`, lanzar `optuna.TrialPruned()`.

In [ ]:
def objective(trial):
    cfg = {**BASE, **search_space(trial)}
    with mlflow.start_run(run_name=f'trial-{trial.number:03d}', nested=True):
        mlflow.log_params(cfg)
        mlflow.set_tag('optuna_trial', trial.number)
        history = []

        def report(epoch, val_loss):
            # TODO 2: trial.report(...) y, si corresponde, raise optuna.TrialPruned()
            raise NotImplementedError('Completa TODO 2')

        t0 = time.time()
        try:
            history, best, _ = fit(cfg, report=report)
        except optuna.TrialPruned:
            mlflow.set_tag('state', 'PRUNED')
            mlflow.log_metric('train_seconds', time.time() - t0)
            raise
        log_history(history)
        mlflow.set_tag('state', 'COMPLETE')
        mlflow.log_metrics({'best_val_loss': best['val_loss'], 'best_val_macro_f1': best['val_macro_f1'],
                            'best_epoch': best['epoch'], 'train_seconds': time.time() - t0})
        return best['val_loss']

## 6. Ejecutar el estudio · 15 min

`N_TRIALS` intentos o `TIMEOUT` segundos, lo que ocurra primero. El muestreador TPE usa los primeros intentos para explorar y luego concentra la búsqueda cerca de las mejores regiones.

In [ ]:
N_TRIALS, TIMEOUT = 20, 15 * 60
sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=6)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner, study_name='covertype-mlp')

t0 = time.time()
with mlflow.start_run(run_name='estudio-optuna') as parent:
    mlflow.log_params({'n_trials': N_TRIALS, 'timeout_s': TIMEOUT, 'sampler': 'TPE', 'pruner': 'Median',
                       'train_size': TRAIN_SIZE})
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)
    states = pd.Series([t.state.name for t in study.trials]).value_counts().to_dict()
    mlflow.log_params({f'best_{k}': v for k, v in study.best_params.items()})
    mlflow.log_metrics({'best_val_loss': study.best_value, 'n_complete': states.get('COMPLETE', 0),
                        'n_pruned': states.get('PRUNED', 0), 'study_seconds': time.time() - t0})
    parent_id = parent.info.run_id
print(f'{len(study.trials)} intentos en {time.time() - t0:.0f} s | estados: {states}')
print('mejor val_loss:', round(study.best_value, 4), '| línea base:', round(best_base['val_loss'], 4))
print('mejores hiperparámetros:', study.best_params)

## 7. Ver y analizar los resultados · 10 min

### En el notebook

In [ ]:
trials = mlflow.search_runs(experiment_names=[EXPERIMENT],
                            filter_string=f"tags.mlflow.parentRunId = '{parent_id}'")
cols = ['tags.mlflow.runName', 'tags.state', 'params.lr', 'params.weight_decay', 'params.dropout', 'params.depth',
        'params.width', 'params.batch_size', 'params.norm', 'metrics.best_val_loss', 'metrics.train_seconds']
table = trials[[c for c in cols if c in trials]].rename(columns=lambda c: c.split('.', 1)[1])
for c in ['lr', 'weight_decay', 'dropout']:
    table[c] = pd.to_numeric(table[c])
table.sort_values('best_val_loss').head(10).style.format(
    {'lr': '{:.1e}', 'weight_decay': '{:.1e}', 'dropout': '{:.2f}', 'best_val_loss': '{:.4f}', 'train_seconds': '{:.0f}'})

In [ ]:
import warnings
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
warnings.filterwarnings('ignore', category=optuna.exceptions.ExperimentalWarning)

for name, plot in [('historia', plot_optimization_history), ('importancia', plot_param_importances)]:
    ax = plot(study)
    fig = ax.figure; fig.set_size_inches(8, 3.6); fig.tight_layout(); plt.show()
    with mlflow.start_run(run_id=parent_id):
        mlflow.log_figure(fig, f'optuna_{name}.png')

done = study.trials_dataframe(attrs=('number', 'value', 'params', 'state')).query("state == 'COMPLETE'")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, (p, log) in zip(axes, [('lr', True), ('weight_decay', True), ('dropout', False)]):
    sc = ax.scatter(done['params_' + p], done['value'], c=done['number'], cmap='viridis', s=45, edgecolor='k', lw=0.4)
    ax.set_xscale('log' if log else 'linear'); ax.set_xlabel(p)
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
axes[0].set_ylabel('mejor val_loss del intento')
fig.colorbar(sc, ax=axes, label='número de intento', pad=0.01)
plt.show()
with mlflow.start_run(run_id=parent_id):
    mlflow.log_figure(fig, 'optuna_slices.png')

### En la interfaz de MLflow

1. Abre la interfaz (celda siguiente en Colab; en Kaggle descarga `mlflow.db` y `mlruns/` y ejecuta `mlflow ui --backend-store-uri sqlite:///mlflow.db` en tu computador).
2. En el experimento **S02-hparam-tuning**, despliega la corrida **estudio-optuna**: dentro están los intentos.
3. Selecciona los intentos → **Compare** → **Parallel Coordinates Plot** con `lr`, `weight_decay`, `dropout`, `depth`, `norm` y la métrica `best_val_loss`.
4. Filtra con `tags.state = 'PRUNED'` para ver qué intentos se detuvieron y en qué época.

In [ ]:
try:
    from google.colab import output
    import subprocess
    subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'sqlite:///mlflow.db', '--port', '5000'])
    time.sleep(5)
    output.serve_kernel_port_as_window(5000)
except ImportError:
    print('No estás en Colab: usa `mlflow ui --backend-store-uri sqlite:///mlflow.db` en una terminal.')

**Pregunta 2.** ¿Cuántos intentos se detuvieron por *pruning* y cuánto tiempo ahorraron (compara `train_seconds` de intentos completos y detenidos)? ¿Hubo algún intento detenido que en tu opinión podía terminar bien?

**Pregunta 3.** Según la importancia de hiperparámetros, ¿cuáles dos dominan? ¿Coincide con lo visto en la sesión (inicialización, normalización, learning rate)?

_Respuestas:_

## 8. Reentrenar la mejor configuración, empaquetar y evaluar en prueba · 5 min

La mejor `val_loss` del estudio es **optimista**: la elegimos entre muchos intentos con el mismo conjunto de validación. Por eso evaluamos **una sola vez** en prueba, con la configuración ganadora reentrenada, y guardamos un artefacto desplegable (pesos + escalador + contrato), igual que en la Práctica B.

In [ ]:
final_cfg = {**BASE, **study.best_params}
with mlflow.start_run(run_name='mejor-receta'):
    mlflow.log_params(final_cfg)
    mlflow.set_tag('source_study', parent_id)
    history, best, best_state = fit(final_cfg)
    log_history(history)
    model = make_mlp(final_cfg).to(device); model.load_state_dict(best_state)
    test = evaluate(model, T['X_test'], T['y_test'])
    mlflow.log_metrics({'best_val_loss': best['val_loss'], **{f'test_{k}': v for k, v in test.items()}})
    torch.save({'cfg': final_cfg, 'state_dict': best_state, 'contract': CONTRACT,
                'mean': torch.tensor(scaler.mean_, dtype=torch.float32),
                'scale': torch.tensor(scaler.scale_, dtype=torch.float32)}, 'best.pt')
    mlflow.log_artifact('best.pt'); mlflow.log_artifact('data_card.json')

summary = pd.DataFrame({'línea base': {'val_loss': best_base['val_loss'], 'val_F1_macro': best_base['val_macro_f1']},
                        'mejor receta': {'val_loss': best['val_loss'], 'val_F1_macro': best['val_macro_f1']}}).T
print(summary.round(4))
print('prueba (una sola vez):', {k: round(v, 4) for k, v in test.items()})

**Pregunta 4.** ¿Cuánto mejoró la búsqueda respecto a la línea base en validación? ¿Por qué esa mejora puede ser optimista, y qué papel cumple la prueba?

**Pregunta 5.** Con el mismo tiempo de cómputo, ¿preferirías 20 intentos con `TRAIN_SIZE = 30_000` o 5 intentos con todo el entrenamiento? Argumenta con lo que viste.

_Respuestas:_

**Entrega:** notebook ejecutado con los dos TODO, `mlflow.db` (con `mlruns/`) o una captura del *Parallel Coordinates Plot* de MLflow, y las respuestas.

**Extensiones:** cambiar el muestreador por `optuna.samplers.RandomSampler` con la misma semilla y comparar la historia de optimización; agregar `act` ∈ {`relu`, `gelu`} al espacio; usar `best_val_macro_f1` como objetivo (`direction='maximize'`).

## Referencias
- Akiba et al. (2019), *Optuna: A Next-generation Hyperparameter Optimization Framework*, KDD.
- Bergstra y Bengio (2012), *Random Search for Hyper-Parameter Optimization*, JMLR.
- [Optuna · documentación](https://optuna.readthedocs.io/) · [MLflow · corridas anidadas y comparación](https://mlflow.org/docs/latest/ml/tracking/).